### Date:  10-15-25

This is dedicated to identifying motifs in the merged pbmc dataset for chrombpnet with tf-modisco-lite: https://github.com/jmschrei/tfmodisco-lite/ which was isntalled in "chromBPnet_tools" conda env w/ pip.
###

We will be using the outputs generated by chromBPnet, more info here: https://github.com/kundajelab/chrombpnet/wiki/Generate-contribution-score-bigwigs
### 
This uses the sequence + chromatin information from the pbmc_merged chromBPnet model  to identify TF motifs. 
###
"The TF-MoDISco algorithm starts with a set of importance scores on genomic sequences and performs the following tasks:
- Identify high-importance windows of the sequences, termed "seqlets"
- Divide the seqlets into positive and negative sets (metaclusters) based on the overall importance score of each seqlet
- Cluster recurring similar selsqlets
- Generate motifs by aligning the clustered seqlets"

###



In [12]:
### Only the counts were used to generate TF-modisco results, took just over a day. The run using profiles ran for voer 3 days before I cancelled the run.

import os 
bash="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/bash"
OUTPUT_DIR="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/modisco_results"


# Run MoDisco motif discovery on both counts and profiles generated by chromBPnet.


# Run for COUNTS (profiels or counts are used, github says it dont matter which)
AVERAGED_SCORES_H5="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc_merged/chrombpnet_model_b1.0/contribution_scores_bw/averaged_scores/averaged_folds_pbmc-merged.counts_scores.h5"
!sbatch /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/bash/run_MoDisco.sh $AVERAGED_SCORES_H5 $OUTPUT_DIR



Submitted batch job 22604009



from the bpnet git:
"While TF-modisco allows one to get a list of sequence motifs, FiNeMo allows one to map the location of these motifs in all the input regions (ex: open chromatin regions). FiNeMO is a GPU-accelerated hit caller for retrieving TFMoDISCo motif occurences from machine-learning-model-generated contribution scores."


This happens in 3 steps:
1) Annotate the motifs in the h5 file generated by TF_MoDisco using MEME formatted JASPAR IDSs. This will generate a list of motifs that are representative of those found in the pbmc data.
2) Using finemo (installed in the chromBPnet_tools environmet) extract the contribution scores for the sequences in open chromatin regions (from the chrombpnet.h5).
3) Using finemo's hit calling algorithm, find the regions/sequences where motifs are found in open chromatin regions.



In [ ]:
# Step 1 
import os
bash="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/bash"

counts_h5="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/modisco_results/tfmodisco_motifs_count_contributions.h5"
counts_out_dir="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/modisco_results/modisco_report_TOMTOM"




!sbatch $bash/modisco_report.sh $counts_h5 $counts_out_dir

Submitted batch job 22603551


In [ ]:
# Step 2
# Now use finemo to extract the motif instances from the counts h5 file:
# This extracts sequences and contributions from ChromBPNet H5 files, using the open chromatin regions (peaks) in the pbmc dataset.

import os

bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/bash"


counts_h5="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc_merged/chrombpnet_model_b1.0/contribution_scores_bw/averaged_scores/averaged_folds_pbmc-merged.counts_scores.h5"

REGIONS_BED="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/peak_calling/outputs/pbmc_merged/pbmc_merged_peaks.final.narrowPeak"

OUTPUT_DIR="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/finemo/pbmc_merged"

!sbatch /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/bash/finemo_extract-h5.sh $counts_h5 $REGIONS_BED $OUTPUT_DIR

Submitted batch job 22597954


In [13]:
# Step 3
# Inital runs using the deafult --cwm-trim-threshold of 0.3 produced few motifs with long lengths 

# Now use finemo to call the hits:
bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/bash"
# MODISCO results (.h5)
MODISCO_H5="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/modisco_results/tfmodisco_motifs_count_contributions.h5"
# finemo extracted scores (.npz), the command uses the NPZ prefix, not the full filename
EXTRACTED_SCORES_NPZ_PREFIX="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/finemo/pbmc_merged/finemo_extracted_counts-scores.npz"
CWM_TRIM_THRESHOLD=0.3
# Output directory
OUTPUT_DIR="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/finemo/pbmc_merged/hits_0.3"

!sbatch $bash/finemo_hitcaller.sh $MODISCO_H5 $EXTRACTED_SCORES_NPZ_PREFIX $CWM_TRIM_THRESHOLD $OUTPUT_DIR 


CWM_TRIM_THRESHOLD=0.7
# Output directory
OUTPUT_DIR="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/finemo/pbmc_merged/hits_0.7"

!sbatch $bash/finemo_hitcaller.sh $MODISCO_H5 $EXTRACTED_SCORES_NPZ_PREFIX $CWM_TRIM_THRESHOLD $OUTPUT_DIR 




Submitted batch job 22604028
Submitted batch job 22604029


In [6]:
# Generate report
import os

HITS_DIR_COUNTS="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/finemo/pbmc_merged"
MODISCO_H5="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/outputs/modisco_results/tfmodisco_motifs_count_contributions.h5"
PEAKS_BED="/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/peak_calling/outputs/pbmc_merged/pbmc_merged_peaks.final.narrowPeak"

!sbatch /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/TF_motif_discovery/bash/finemo_report.sh $HITS_DIR_COUNTS $MODISCO_H5 $PEAKS_BED 


Submitted batch job 22604751
